<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v2/blob/master/KD_Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10 · Knowledge Distillation — Main Experiment

Full KD experiment with selected teacher and student, across 3 seeds.
Compares two student initialization strategies:

- **KD-FT**: student initialized from pretrained baseline, then KD training
- **KD-Scratch**: student initialized randomly, then KD training

**Hyperparameters:** T=4.0, alpha=0.7 (justified by ablation in notebook 09).

**Expected outcome:** KD-FT outperforms KD-Scratch because the student already has
useful pretrained features before distillation begins.


In [7]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/thesis/utils/")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ utils loaded from Drive


In [8]:
# ── Standard imports ────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import (
    VGG_Scratch, VGG_Pretrained,
    ResNet_Scratch, ResNet18_Pretrained,
    MobileNetV2_Scratch, MobileNetV3_Pretrained,
    count_params, model_size_mb, MODEL_REGISTRY,
)
from utils.train import (
    setup_device, set_seed, evaluate,
    train_model, train_model_three_phase,
    train_multi_seed, train_kd, plot_history,
)

device = setup_device(seed=41)


Device: cuda


In [9]:
# ── Dataset setup ───────────────────────────────────────────────────
prepare_dataset()


1/4 Download
✅ VWW archive already downloaded
2/4 Extract
✅ VWW already extracted
3/4 Find root
   Root: /content/vww_work/extracted/vw_coco2014_96
4/4 Manifests
✅ Manifests already exist: /content/drive/My Drive/vww_fixed_split_manifests


PosixPath('/content/vww_work/extracted/vw_coco2014_96')

In [10]:
SAVE_DIR = "/content/drive/My Drive/Colab Notebooks"


In [11]:
# ── Set after running notebooks 04, 07, and 09 ───────────────────────
TEACHER_CKPT     = f"{SAVE_DIR}/vgg_pretrained_seed_63.pth"
STUDENT_CKPT     = f"{SAVE_DIR}/mobilenetv2_baseline_seed_41.pth"
TEACHER_MODEL_FN = VGG_Pretrained
STUDENT_MODEL_FN = MobileNetV2_Scratch

KD_TEMPERATURE = 4.0
KD_ALPHA       = 0.7
KD_EPOCHS      = 30
KD_LR          = 1e-3
KD_SEEDS       = [41, 52, 63]


In [12]:
train_loader, val_loader = get_loaders(batch_size=64, augmentation="minimal")

teacher = TEACHER_MODEL_FN().to(device)
teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=device))
teacher_acc = evaluate(teacher, val_loader, device)
print(f"Teacher val accuracy: {teacher_acc*100:.2f}%")


Train: 7000 | Val: 1500 | Batch: 64
Downloading: "https://download.pytorch.org/models/vgg16_bn-6c64b313.pth" to /root/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth


100%|██████████| 528M/528M [00:06<00:00, 80.0MB/s]


Teacher val accuracy: 88.07%


In [13]:
# ── KD-FT: initialize student from pretrained baseline ───────────────
print("\n" + "="*55 + "\nKD-FT (fine-tune from pretrained baseline)\n" + "="*55)
kd_ft_results = []

for seed in KD_SEEDS:
    print(f"\nSeed {seed}")
    set_seed(seed)
    student = STUDENT_MODEL_FN().to(device)
    student.load_state_dict(torch.load(STUDENT_CKPT, map_location=device))
    baseline_acc = evaluate(student, val_loader, device)

    best_acc, elapsed = train_kd(
        student=student, teacher=teacher,
        train_loader=train_loader, val_loader=val_loader, device=device,
        epochs=KD_EPOCHS, lr=KD_LR, weight_decay=1e-4,
        temperature=KD_TEMPERATURE, alpha=KD_ALPHA,
        patience=10, save_path=f"{SAVE_DIR}/kd_ft_seed_{seed}.pth",
    )
    kd_ft_results.append({
        "seed": seed, "baseline": baseline_acc,
        "kd_acc": best_acc, "delta": best_acc - baseline_acc, "elapsed": elapsed
    })



KD-FT (fine-tune from pretrained baseline)

Seed 41
Initial student accuracy: 78.80%
Epoch   1/30 | Train 81.24% | Val 76.00%
Epoch   2/30 | Train 81.43% | Val 74.07%
Epoch   3/30 | Train 82.06% | Val 76.07%
Epoch   4/30 | Train 82.41% | Val 77.33%
Epoch   5/30 | Train 82.57% | Val 76.93%
Epoch   6/30 | Train 82.99% | Val 75.67%
Epoch   7/30 | Train 83.63% | Val 75.53%
Epoch   8/30 | Train 83.49% | Val 75.80%
Epoch   9/30 | Train 83.86% | Val 76.93%
Epoch  10/30 | Train 84.50% | Val 75.60%
🛑 Early stopping at epoch 10

✅ Best KD val acc: 78.80%  (1.9 min)

Seed 52
Initial student accuracy: 78.80%
Epoch   1/30 | Train 81.40% | Val 75.33%
Epoch   2/30 | Train 82.07% | Val 76.07%
Epoch   3/30 | Train 81.76% | Val 76.93%
Epoch   4/30 | Train 82.84% | Val 76.07%
Epoch   5/30 | Train 81.96% | Val 76.47%
Epoch   6/30 | Train 82.87% | Val 76.67%
Epoch   7/30 | Train 83.14% | Val 76.60%
Epoch   8/30 | Train 82.99% | Val 76.80%
Epoch   9/30 | Train 83.94% | Val 78.33%
Epoch  10/30 | Train 84.76

In [14]:
# ── KD-Scratch: initialize student from random weights ───────────────
print("\n" + "="*55 + "\nKD-Scratch (random initialization)\n" + "="*55)
kd_scratch_results = []

for seed in KD_SEEDS:
    print(f"\nSeed {seed}")
    set_seed(seed)
    student = STUDENT_MODEL_FN().to(device)   # fresh random init — do NOT load baseline
    scratch_acc = evaluate(student, val_loader, device)
    print(f"  Random init accuracy: {scratch_acc*100:.2f}%")

    best_acc, elapsed = train_kd(
        student=student, teacher=teacher,
        train_loader=train_loader, val_loader=val_loader, device=device,
        epochs=KD_EPOCHS, lr=KD_LR, weight_decay=1e-4,
        temperature=KD_TEMPERATURE, alpha=KD_ALPHA,
        patience=10, save_path=f"{SAVE_DIR}/kd_scratch_seed_{seed}.pth",
    )
    kd_scratch_results.append({
        "seed": seed, "baseline": scratch_acc,
        "kd_acc": best_acc, "delta": best_acc - scratch_acc, "elapsed": elapsed
    })



KD-Scratch (random initialization)

Seed 41
  Random init accuracy: 50.00%
Initial student accuracy: 50.00%
Epoch   1/30 | Train 61.77% | Val 65.53% ✅
Epoch   2/30 | Train 67.69% | Val 67.73% ✅
Epoch   3/30 | Train 70.00% | Val 63.33%
Epoch   4/30 | Train 70.96% | Val 69.33% ✅
Epoch   5/30 | Train 72.61% | Val 72.20% ✅
Epoch   6/30 | Train 74.17% | Val 70.27%
Epoch   7/30 | Train 74.76% | Val 73.07% ✅
Epoch   8/30 | Train 75.43% | Val 73.67% ✅
Epoch   9/30 | Train 75.90% | Val 73.93% ✅
Epoch  10/30 | Train 76.36% | Val 70.67%
Epoch  11/30 | Train 76.87% | Val 74.27% ✅
Epoch  12/30 | Train 77.31% | Val 75.40% ✅
Epoch  13/30 | Train 77.93% | Val 74.20%
Epoch  14/30 | Train 78.40% | Val 76.27% ✅
Epoch  15/30 | Train 78.93% | Val 75.07%
Epoch  16/30 | Train 79.44% | Val 76.40% ✅
Epoch  17/30 | Train 80.04% | Val 77.13% ✅
Epoch  18/30 | Train 80.36% | Val 77.27% ✅
Epoch  19/30 | Train 81.41% | Val 76.27%
Epoch  20/30 | Train 81.80% | Val 75.00%
Epoch  21/30 | Train 81.80% | Val 76.47%
Epoc

In [16]:
# ── Summary ──────────────────────────────────────────────────────────
import pandas as pd

def summarize(results, label):
    accs = [r["kd_acc"] for r in results]
    return {
        "Condition": label,
        "Mean Acc (%)": round(np.mean(accs)*100, 2),
        "Std (%)":      round(np.std(accs)*100,  2),
        "Best Acc (%)": round(max(accs)*100,      2),
        "Δ vs baseline (%)": round((np.mean(accs) - np.mean([r["baseline"] for r in results]))*100, 2),
    }

student_base_acc = evaluate(
    STUDENT_MODEL_FN().to(device).__class__(
        **{}
    ), val_loader, device
) if False else None  # placeholder — filled from notebook 07 results

rows = [
    {"Condition": "Student baseline (no KD)",
     "Mean Acc (%)": "→ see nb 07", "Std (%)": "—", "Best Acc (%)": "—", "Δ vs baseline (%)": "—"},
    summarize(kd_ft_results,     "KD-FT"),
    summarize(kd_scratch_results,"KD-Scratch"),
    {"Condition": "Teacher (VGG16-BN pretrained)",
     "Mean Acc (%)": round(teacher_acc*100, 2), "Std (%)": "—", "Best Acc (%)": "—", "Δ vs baseline (%)": "—"},
]

df = pd.DataFrame(rows).set_index("Condition")
print("\n" + df.to_string())
print("\nBest KD-FT checkpoint:     ", f"{SAVE_DIR}/kd_ft_seed_{max(kd_ft_results, key=lambda r:r['kd_acc'])['seed']}.pth")
print("Best KD-Scratch checkpoint:", f"{SAVE_DIR}/kd_scratch_seed_{max(kd_scratch_results, key=lambda r:r['kd_acc'])['seed']}.pth")



                              Mean Acc (%) Std (%) Best Acc (%) Δ vs baseline (%)
Condition                                                                        
Student baseline (no KD)       → see nb 07       —            —                 —
KD-FT                                79.09    0.41        79.67              0.29
KD-Scratch                           77.18    0.28        77.47             27.18
Teacher (VGG16-BN pretrained)        88.07       —            —                 —

Best KD-FT checkpoint:      /content/drive/My Drive/Colab Notebooks/kd_ft_seed_63.pth
Best KD-Scratch checkpoint: /content/drive/My Drive/Colab Notebooks/kd_scratch_seed_52.pth
